<a href="https://colab.research.google.com/github/kou1-n/Home_Page/blob/master/%E5%8B%95%E7%94%BB%E3%83%95%E3%83%AC%E3%83%BC%E3%83%A0%E6%8A%BD%E5%87%BA4colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-image opencv-python-headless img2pdf tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# -*- coding: utf-8 -*-
# 依存ライブラリのインストール (Colab環境で最初に実行)

import sys
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
import img2pdf
import traceback
from tqdm.notebook import tqdm  # ColabのNotebook用tqdm
from google.colab import files # ファイルアップロード用
import shutil # ディレクトリ操作用

def process_video(video_path, out_dir):
    """
    動画ファイルを処理して、変化があったフレームを抽出し、PDFにまとめる関数

    Args:
        video_path (str): 入力動画ファイルのパス
        out_dir (str): 抽出したフレームとPDFを保存するディレクトリのパス
    """
    # 出力ディレクトリ作成 (存在しない場合)
    os.makedirs(out_dir, exist_ok=True)

    # 入力チェック
    if not os.path.isfile(video_path):
        print(f"[エラー] 動画ファイルが見つかりません: {video_path}")
        return
    if not os.path.isdir(out_dir):
        print(f"[エラー] 保存先フォルダが不正です: {out_dir}")
        return

    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            print(f"[エラー] 動画を開けません: {video_path}")
            return

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            print("[エラー] 動画のフレーム数を取得できませんでした。ファイルが破損している可能性があります。")
            cap.release()
            return

        saved_jpg_list = []
        prev_gray = None
        slide_count = 0
        frame_idx = 0

        # tqdmで進捗を表示
        pbar = tqdm(total=total_frames, desc="フレーム処理中")

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1
            pbar.update(1) # プログレスバー更新

            # グレースケール変換
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # 最初のフレームは保存
            if prev_gray is None:
                slide_count += 1
                jpg_path = save_frame(frame, cap, out_dir, slide_count)
                if jpg_path:
                    saved_jpg_list.append(jpg_path)
                prev_gray = gray
                continue

            # SSIM計算 ( Structural Similarity Index )
            # 画像間の構造的な類似度を測る指標。1に近いほど似ている。
            score, _ = ssim(prev_gray, gray, full=True) # diffは使わないので_で受け取る

            # 差分絶対値の計算と差分ピクセル比率の計算
            # フレーム間のピクセル値の差を計算し、変化の度合いを見る。
            diff_abs = cv2.absdiff(gray, prev_gray)
            # 閾値(ここでは10)以上の差があるピクセルの数を数える
            # diff_nonzero = np.count_nonzero(diff_abs > 10) # 少しの変化を無視する場合
            diff_nonzero = np.count_nonzero(diff_abs) # わずかな変化も検出する場合
            diff_ratio = diff_nonzero / diff_abs.size # 全ピクセル数に対する変化ピクセルの割合

            # 新しいスライド（変化が大きいフレーム）と判定する閾値
            # SSIMが0.92未満（類似度が低い）かつ 差分ピクセル比率が1%より大きい場合に変化とみなす
            # これらの値は動画の内容によって調整が必要な場合がある
            ssim_threshold = 0.92
            diff_ratio_threshold = 0.01

            if score < ssim_threshold and diff_ratio > diff_ratio_threshold:
                slide_count += 1
                jpg_path = save_frame(frame, cap, out_dir, slide_count)
                if jpg_path:
                   saved_jpg_list.append(jpg_path)
                prev_gray = gray
            # else:
                # 変化が小さいフレームはスキップ
                # pass

        cap.release()
        pbar.close()
        print("フレーム処理完了。")

        if not saved_jpg_list:
            print("[警告] 抽出されたフレームがありません。閾値の調整が必要かもしれません。")
            return

        # PDF生成
        pdf_path = os.path.join(out_dir, "output.pdf")
        print(f"PDFを生成中: {pdf_path}")
        try:
            # img2pdfにファイルパスのリストを渡す
            with open(pdf_path, "wb") as f:
                # サイズの大きな画像を扱う場合、メモリ使用量を抑えるために
                # 画像データを直接渡すのではなく、ファイルパスを渡す方が効率的
                f.write(img2pdf.convert(saved_jpg_list))
            print("PDF生成完了。")
            # Colab環境からファイルをダウンロードする場合
            # files.download(pdf_path)
            # print(f"{pdf_path} をダウンロードします。")

        except Exception as e:
            print(f"[エラー] PDF生成中にエラーが発生しました: {e}")
            traceback.print_exc()
            # エラー発生時も、生成された画像ファイルは残る

    except Exception as e:
        print(f"[エラー] 動画処理中に予期せぬエラーが発生しました: {e}")
        traceback.print_exc()
    finally:
        if 'cap' in locals() and cap.isOpened():
            cap.release()
        if 'pbar' in locals():
            pbar.close()


def save_frame(frame, cap, out_dir, slide_count):
    """
    フレーム画像にタイムスタンプを描画し、JPEGファイルとして保存する関数

    Args:
        frame (numpy.ndarray): 保存するフレーム画像 (BGR形式)
        cap (cv2.VideoCapture): 動画キャプチャオブジェクト
        out_dir (str): 保存先ディレクトリ
        slide_count (int): スライド番号（ファイル名に使用）

    Returns:
        str or None: 保存したJPEGファイルのパス。保存に失敗した場合はNone。
    """
    try:
        # タイムスタンプ取得 (ミリ秒)
        ms = cap.get(cv2.CAP_PROP_POS_MSEC)
        if ms < 0: # タイムスタンプが取得できない場合がある
             ms = 0 # 代わりに0を使うなど考慮

        sec = int(ms // 1000)
        hh = sec // 3600
        mm = (sec % 3600) // 60
        ss = sec % 60
        ts_text = f"{hh:02}:{mm:02}:{ss:02}"

        # タイムスタンプ描画設定
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 1.0
        color = (0, 255, 0)  # BGR形式で緑色
        thickness = 2
        line_type = cv2.LINE_AA
        # 画像の左下に描画
        text_pos = (10, frame.shape[0] - 10) # (x座標, y座標)

        # 描画するテキストと設定を使って画像に書き込む
        cv2.putText(frame, ts_text, text_pos, font, font_scale, color, thickness, line_type)

        # 保存ファイル名生成 (例: frame_00001_00-00-15.jpg)
        file_name = f"frame_{slide_count:05d}_{hh:02d}-{mm:02d}-{ss:02d}.jpg"
        save_path = os.path.join(out_dir, file_name)

        # JPEGファイルとして保存
        # cv2.imwrite() は成功した場合 True を返す
        success = cv2.imwrite(save_path, frame)
        if success:
            # print(f"フレーム保存: {save_path}") # 詳細ログが必要な場合
            return save_path
        else:
            print(f"[警告] フレームの保存に失敗しました: {save_path}")
            return None
    except Exception as e:
        print(f"[エラー] フレーム保存中にエラーが発生しました: {e}")
        traceback.print_exc()
        return None


# --- Google Colabでの実行 ---

# 1. Google Driveをマウント (オプション)
#from google.colab import drive
#drive.mount('/content/drive')

# 2. 動画ファイルの準備

#   方法A: Google Driveからファイルを指定
#video_file_path = '/content/drive/MyDrive/情報環境概論I_第2回マルチメディア情報処理 ビデオストリーム.mp4' # <-- ★要変更: あなたの動画ファイルパス

#   方法B: ローカルからファイルをアップロード
# print("動画ファイルをアップロードしてください:")
# uploaded = files.upload()
# if not uploaded:
#   raise Exception("ファイルがアップロードされませんでした。")
# video_file_path = list(uploaded.keys())[0] # アップロードされた最初のファイル名を取得
# print(f"アップロードされたファイル: {video_file_path}")

#   方法C: サンプル動画を使う場合 (例: OpenCVのサンプルなど、もしあれば)
# video_file_path = 'path/to/sample/video.mp4' # <-- ★要変更

# === ★★★ ここで動画ファイルのパスを指定してください ★★★ ===
# 例:
# video_file_path = '/content/drive/MyDrive/Colab Notebooks/input_video.mp4' # Google Driveの場合
# video_file_path = 'uploaded_video.mp4' # ローカルからアップロードした場合など

# このセルを実行する前に、上のコメント部分を参考に video_file_path を設定してください。
# もし方法Bでアップロードする場合、このセルを実行する前に下のコメントアウトを解除し、
# 実行時に表示されるボタンからファイルをアップロードしてください。
# video_file_path = None # 未設定状態を示す
# print("動画ファイルをアップロードしてください:")
# uploaded = files.upload()
# if uploaded:
#   video_file_path = list(uploaded.keys())[0]
#   print(f"ファイル '{video_file_path}' を使用します。")
# else:
#   print("ファイルがアップロードされませんでした。処理を中断します。")
#   # もしDriveを使いたい場合は、ここの処理をDriveのパス指定に切り替えてください。
#   # video_file_path = '/content/drive/MyDrive/your_video.mp4' # 例

# ダミーのパス（エラーを防ぐため、必ず上で有効なパスに設定してください）
video_file_path = '/content/drive/MyDrive/情報環境概論I_第2回マルチメディア情報処理 ビデオストリーム.mp4' # <-- ★★★ 必ず実際のパスに変更してください ★★★


# 3. 出力先フォルダの指定
output_directory = '/content/output_slides' # Colabの一時ストレージ内に出力

# 古い出力フォルダがあれば削除（必要に応じて）
# if os.path.exists(output_directory):
#    print(f"既存の出力フォルダ {output_directory} を削除します。")
#    shutil.rmtree(output_directory)

# 4. 動画処理の実行
if video_file_path != 'PLEASE_SET_VIDEO_PATH' and os.path.exists(video_file_path):
    print(f"動画ファイル: {video_file_path}")
    print(f"出力先フォルダ: {output_directory}")
    process_video(video_file_path, output_directory)

    # 5. 結果の確認とダウンロード (オプション)
    print("\n処理が完了しました。")
    print(f"抽出された画像とPDFは {output_directory} に保存されています。")

    # 生成されたPDFファイルをダウンロードする場合
    pdf_file = os.path.join(output_directory, "output.pdf")
    if os.path.exists(pdf_file):
        print(f"\nPDFファイル '{pdf_file}' をダウンロードします...")
        try:
            files.download(pdf_file)
        except Exception as e:
            print(f"ダウンロード中にエラーが発生しました: {e}")
    else:
        print(f"\nPDFファイル {pdf_file} が見つかりませんでした。")

    # 生成された画像ファイルをzipでまとめてダウンロードする場合 (ファイル数が多い場合に便利)
    # import shutil
    # zip_filename = '/content/output_slides.zip'
    # if os.path.exists(output_directory) and len(os.listdir(output_directory)) > 1: # PDF以外にもファイルがあるか確認
    #   print(f"\n出力フォルダをZIPファイル '{zip_filename}' にまとめてダウンロードします...")
    #   try:
    #       shutil.make_archive(zip_filename.replace('.zip', ''), 'zip', output_directory)
    #       files.download(zip_filename)
    #   except Exception as e:
    #       print(f"ZIP作成またはダウンロード中にエラーが発生しました: {e}")
    # else:
    #    print("\n画像ファイルが見つからないため、ZIPファイルは作成しません。")

else:
    if video_file_path == 'PLEASE_SET_VIDEO_PATH':
        print("[エラー] 動画ファイルのパス (video_file_path) を設定してください。")
    else:
        print(f"[エラー] 指定された動画ファイルが見つかりません: {video_file_path}")

動画ファイル: /content/drive/MyDrive/情報環境概論I_第2回マルチメディア情報処理 ビデオストリーム.mp4
出力先フォルダ: /content/output_slides


フレーム処理中:   0%|          | 0/49884 [00:00<?, ?it/s]

フレーム処理完了。
PDFを生成中: /content/output_slides/output.pdf
PDF生成完了。

処理が完了しました。
抽出された画像とPDFは /content/output_slides に保存されています。

PDFファイル '/content/output_slides/output.pdf' をダウンロードします...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>